# DS/CMPSC 410 MiniProject Deliverable #2

# Spring 2025
### Instructor: Prof. John Yen
### TA: Peng Jin and Jingxi Zhu

### Learning Objectives
- Be able to represent ports scanned by scanners as binary features using One Hot Encoding
- Be able to apply k-means clustering to cluster the scanners based on the set of ports they scanned. 
- Be able to identify the set of top k ports for one-hot encoding ports scanned.
- Be able to interpret the results of clustering using cluster centers.
- After successful clustering of the small Darknet dataset, conduct clustering on the large Darknet dataset (running spark in cluster mode).
- Be able to evaluate the result of k-means clustering (cluster mode) using Silhouette score and Mirai labels.
- Be able to use .persist() and .unpersist() to improve the scalability/efficiency of PySpark code.

### Total points: 120 
- Problem 1A: 5 points
- Problem 1B: 5 points
- Problem 1C: 10 points
- Problem 2: 10 points 
- Problem 3: 10 points 
- Problem 4: 5 points
- Problem 5: 10 points
- Problem 6: 10 points
- Problem 7: 15 points
- Problem 8: 40 points

### Items for Submission: 
- Completed Jupyter Notebook for local mode (HTML format)
- .py file for successful execution in cluster mode 
- log file (including execution time information) for successful execution in cluster mode
- The csv file (generated in cluster mode) for Mirai Ratio and Cluster Centers for all clusters
- The csv file (generated in cluster mode) for sorted count of scanners that scan the same number of ports
- The first data file (i.e., part-00000) (generated in cluster mode) in ``sorted_top_ports_counts.txt``
  
### Due: 11:59 pm, April 11, 2025
### Early Submission bonus (before midnight April 6, 2025): 12 points

In [1]:
import pyspark
import csv

In [2]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, LongType, IntegerType, DecimalType, BooleanType
from pyspark.sql.functions import col, column
from pyspark.sql.functions import expr
from pyspark.sql.functions import split
from pyspark.sql.functions import array_contains
from pyspark.sql import Row
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler, IndexToString
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [3]:
ss = SparkSession.builder.master("local").appName("MiniProject 2 k-meas Clustering using OHE").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/02 14:25:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
ss.sparkContext.setLogLevel("WARN")

# Problem 1A (5 points)
Complete the path for input file in the code below and enter your name in this Markdown cell:
- Name: Aidan Vesci
## Note: You will need to change the name of the input file in the cluster mode to `Day_2020_profile.csv`

In [5]:
scanner_schema = StructType([StructField("_c0", IntegerType(), False), \
                             StructField("id", IntegerType(), False ), \
                             StructField("numports", IntegerType(), False), \
                             StructField("lifetime", DecimalType(), False ), \
                             StructField("Bytes", IntegerType(), False ), \
                             StructField("Packets", IntegerType(), False), \
                             StructField("average_packetsize", IntegerType(), False), \
                             StructField("MinUniqueDests", IntegerType(), False),\
                             StructField("MaxUniqueDests", IntegerType(), False), \
                             StructField("MinUniqueDest24s", IntegerType(), False), \
                             StructField("MaxUniqueDest24s", IntegerType(), False), \
                             StructField("average_lifetime", DecimalType(), False), \
                             StructField("mirai", BooleanType(), True), \
                             StructField("zmap", BooleanType(), True),
                             StructField("masscan", BooleanType(), True),
                             StructField("country", StringType(), False), \
                             StructField("traffic_types_scanned_str", StringType(), False), \
                             StructField("ports_scanned_str", StringType(), False), \
                             StructField("host_tags_per_censys", StringType(), False), \
                             StructField("host_services_per_censys", StringType(), False) \
                           ])

In [6]:
Scanners_df = ss.read.csv("/storage/home/ajv5723/work/MiniProj2/sampled_profile.csv", schema= scanner_schema, header= True, inferSchema=False )

## We can use printSchema() to display the schema of the DataFrame Scanners_df to see whether it was consistent with the schema.

In [7]:
Scanners_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- numports: integer (nullable = true)
 |-- lifetime: decimal(10,0) (nullable = true)
 |-- Bytes: integer (nullable = true)
 |-- Packets: integer (nullable = true)
 |-- average_packetsize: integer (nullable = true)
 |-- MinUniqueDests: integer (nullable = true)
 |-- MaxUniqueDests: integer (nullable = true)
 |-- MinUniqueDest24s: integer (nullable = true)
 |-- MaxUniqueDest24s: integer (nullable = true)
 |-- average_lifetime: decimal(10,0) (nullable = true)
 |-- mirai: boolean (nullable = true)
 |-- zmap: boolean (nullable = true)
 |-- masscan: boolean (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_types_scanned_str: string (nullable = true)
 |-- ports_scanned_str: string (nullable = true)
 |-- host_tags_per_censys: string (nullable = true)
 |-- host_services_per_censys: string (nullable = true)



# In this lab, our goal is to answer the question:
## Q: What groups of scanners are similar in the ports they scan?

### Because we know (from MiniProject 1) about two third of the scanners scan only 1 port, we can exclude them so that we focus on the grouping of those scanners that scan at least two ports.

### Because the feature `numports` record the total number of ports being scanned by each scanner, we can use it to separate 1-port-scanners from multi-port-scanners.

In [8]:
one_port_scanners = Scanners_df.where(col('numports') == 1)

In [9]:
one_port_scanners.show(3)

25/04/02 14:25:36 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
 Schema: _c0, id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
Expected: _c0 but found: 
CSV file: file:///storage/home/ajv5723/work/MiniProj2/sampled_profile.csv


+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+-----------------+--------------------+------------------------+
|    _c0|     id|numports|lifetime|Bytes|Packets|average_packetsize|MinUniqueDests|MaxUniqueDests|MinUniqueDest24s|MaxUniqueDest24s|average_lifetime|mirai| zmap|masscan|country|traffic_types_scanned_str|ports_scanned_str|host_tags_per_censys|host_services_per_censys|
+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+-----------------+--------------------+------------------------+
|1645181|1645181|       1|       0|   60|      1|                60|             1|             1|               1|               1|               0|false|false|  false|     BR|                   

In [10]:
multi_port_scanners = Scanners_df.where(col("numports") > 1)

In [11]:
multi_port_scanners_count = multi_port_scanners.count()

In [12]:
print(multi_port_scanners_count)

73663


# Before clustering, it can be useful to understand the distribution of scanners based on the number of ports they scan.
# Because the column "numports' already contain the information about the number of ports each scanner (represented by a row in the input csv file), we can use `groupby("numports")` on `Scanners_df` followed by `.count()` 

In [13]:
ScannersCount_byNumPorts = Scanners_df.groupby("numports").count()

In [14]:
SortedScannersCount_byNumPorts= ScannersCount_byNumPorts.orderBy("count", ascending=False)

In [15]:
SortedScannersCount_byNumPorts.show(10)

+--------+------+
|numports| count|
+--------+------+
|       1|153399|
|       2| 24114|
|       3| 16206|
|       4|  5952|
|       5|  4400|
|       6|  3392|
|       7|  2924|
|       8|  2779|
|       9|  2669|
|      11|  2500|
+--------+------+
only showing top 10 rows



In [16]:
output1 = "/storage/home/ajv5723/work/MiniProj2/local/SortedScannersCount_byNumPorts.csv"
SortedScannersCount_byNumPorts.write.option("header", True).csv(output1)

# What is the maximum and average of number of ports being scanned?

# Problem 1B (5 points)
## Use `agg` method of DataFrame to find the maximal and average number of ports being scanned across all scanners.

In [16]:
MaxNumPorts = Scanners_df.agg({"numports" : "max"})

In [17]:
MaxNumPorts.show(1)

+-------------+
|max(numports)|
+-------------+
|        65386|
+-------------+



In [18]:
AvgNumPorts = Scanners_df.agg({"numports" : "avg"})

In [19]:
AvgNumPorts.show(1)

+-----------------+
|    avg(numports)|
+-----------------+
|6.280645814799482|
+-----------------+



# We can also find scanners that scan many ports by investigating scanners whose numports is unique (one-of-a-kind, a unicorn)

In [20]:
ScannersCount_byNumPorts.where(col("count")==1).show(10)

+--------+-----+
|numports|count|
+--------+-----+
|    1238|    1|
|   31161|    1|
|      85|    1|
|    2247|    1|
|     362|    1|
|   35133|    1|
|    6419|    1|
|     115|    1|
|   36794|    1|
|    4843|    1|
+--------+-----+
only showing top 10 rows



# We noticed that some of the scanners that scan for very large number of ports (we call them Extreme Scanners) is unique in the number of ports they scan.
## A heuristic to separate extreme scanners: Find the largest number of ports that are scanned by at least two scanners. Use the number as the threshold to filter extreme scanners.

In [21]:
non_rare_NumPorts = SortedScannersCount_byNumPorts.where(col("count") > 1)

In [22]:
non_rare_NumPorts.show(3)

+--------+------+
|numports| count|
+--------+------+
|       1|153399|
|       2| 24114|
|       3| 16206|
+--------+------+
only showing top 3 rows



# DataFrame can aggregate a column using .agg({ "column name" : "operator name" })
## We can find the maximum of numports column using "max" as aggregation operator.
## The result is a DataFrame with only column named as ``<operator name>(<column name>)``

In [23]:
max_non_rare_NumPorts_df = non_rare_NumPorts.agg({"numports" : "max"})
max_non_rare_NumPorts_df.show()

+-------------+
|max(numports)|
+-------------+
|          654|
+-------------+



# We want to record this number, rather than using the number (654) as a constant in the code below.
## Why?
## Because the number is based on the data, which is different for the cluster mode.

In [24]:
max_non_rare_NumPorts_rdd = max_non_rare_NumPorts_df.rdd.map(lambda x: x[0])
max_non_rare_NumPorts_rdd.take(2)

[654]

In [25]:
max_non_rare_NumPorts_list = max_non_rare_NumPorts_rdd.collect()
print(max_non_rare_NumPorts_list)

[654]


In [26]:
max_non_rare_NumPorts=max_non_rare_NumPorts_list[0]
print(max_non_rare_NumPorts)

654


## We are going to focus on the grouping of scanners that scan at least two ports, and do not scan extremely large number of ports. We will call these scanners Non-extreme Multi-port Scanners.
## We will save the extreme scanners in a csv file so that we can process it separately.

In [27]:
extreme_scanners = Scanners_df.where(col("numports") > max_non_rare_NumPorts)

In [29]:
path2="/storage/home/ajv5723/work/MiniProj2/local/Extreme_Scanners.csv"
extreme_scanners.write.option("header",True).csv(path2)

25/04/01 21:06:39 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
 Schema: _c0, id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
Expected: _c0 but found: 
CSV file: file:///storage/home/ajv5723/work/MiniProj2/sampled_profile.csv


In [28]:
non_extreme_multi_port_scanners = Scanners_df.where(col("numports") <= max_non_rare_NumPorts).where(col("numports") > 1)

In [29]:
non_extreme_multi_port_scanners.persist()

DataFrame[_c0: int, id: int, numports: int, lifetime: decimal(10,0), Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: decimal(10,0), mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string]

In [30]:
non_extreme_multi_port_scanners.count()

25/04/02 14:26:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
 Schema: _c0, id, numports, lifetime, Bytes, Packets, average_packetsize, MinUniqueDests, MaxUniqueDests, MinUniqueDest24s, MaxUniqueDest24s, average_lifetime, mirai, zmap, masscan, country, traffic_types_scanned_str, ports_scanned_str, host_tags_per_censys, host_services_per_censys
Expected: _c0 but found: 
CSV file: file:///storage/home/ajv5723/work/MiniProj2/sampled_profile.csv


73599

# Part A: One Hot Encoding of Top 100 Ports
We want to apply one hot encoding to the top 100 ports scanned by scanners. 
- A1: Find top k ports scanned by non_extreme_multi_port scanners (This is similar to the first part of MiniProject 1)
- A2: Generate One Hot Encodding for these top k ports

In [31]:
non_extreme_multi_port_scanners.select("ports_scanned_str").show(4)

+--------------------+
|   ports_scanned_str|
+--------------------+
|         17128-17136|
|17128-17130-17132...|
|23-80-81-1023-232...|
|17128-17132-17136...|
+--------------------+
only showing top 4 rows



# For each port scanned, count the Total Number of Scanners that Scan the Given Port
Like MiniProject 1, to calculate this, we need to 
- (a) convert the ports_scanned_str into an array/list of ports
- (b) Convert the DataFrame into an RDD
- (c) Use flatMap to count the total number of scanners for each port.

# The Following Code Implements the three steps.
## (a) Create a new column "Ports_Array" by splitting the column "ports_scanned_str" using "-" as the delimiter.

In [32]:
# (a)
NEMP_Scanners_df=non_extreme_multi_port_scanners.withColumn("Ports_Array", split(col("ports_scanned_str"), "-") )
NEMP_Scanners_df.show(2)

+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+--------------------+--------------------+------------------------+--------------------+
|    _c0|     id|numports|lifetime|Bytes|Packets|average_packetsize|MinUniqueDests|MaxUniqueDests|MinUniqueDest24s|MaxUniqueDest24s|average_lifetime|mirai| zmap|masscan|country|traffic_types_scanned_str|   ports_scanned_str|host_tags_per_censys|host_services_per_censys|         Ports_Array|
+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+--------------------+--------------------+------------------------+--------------------+
|2091467|2091467|       2|     200|  752|     12|                62|             1|             1|               1|         

# We will need to use NEMP_Scanners_df multiple times in creating OHE features later, so we persist it.

In [33]:
NEMP_Scanners_df.persist()

DataFrame[_c0: int, id: int, numports: int, lifetime: decimal(10,0), Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: decimal(10,0), mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>]

## (b) We convert the column ```Ports_Array``` into an RDD so that we can apply flatMap for counting the number of scanners, among those that scan at least two ports, but not extreme scanners, that scan each port.

In [34]:
Ports_Scanned_RDD = NEMP_Scanners_df.select("Ports_Array").rdd

In [35]:
Ports_Scanned_RDD.take(5)

25/04/02 14:26:38 WARN BlockManager: Task 24 already completed, not releasing lock for rdd_99_0


[Row(Ports_Array=['17128', '17136']),
 Row(Ports_Array=['17128', '17130', '17132', '17134', '17136', '17138', '17140']),
 Row(Ports_Array=['23', '80', '81', '1023', '2323', '5555', '7574', '8080', '8443', '37215', '49152', '52869']),
 Row(Ports_Array=['17128', '17132', '17136', '17140', '17142', '34230']),
 Row(Ports_Array=['137', '17130'])]

## (c) Because each port number in the Ports_Array column for each row/scanner occurs only once, we can count the total number of scanners by counting the total occurance of each port number through flatMap.
### Because each element of the ``Ports_Scanned_RDD`` rdd is a Row object, we need to first extract ``Ports_Array`` from the row object.
# Problem 1C (10%) Complete the code below to count the total number of scanners that scan a port using ``flatMap`` and ``reduceByKey`` (like miniProject 1).

In [36]:
Ports_Scanned_RDD.take(3)

25/04/02 14:26:41 WARN BlockManager: Task 25 already completed, not releasing lock for rdd_99_0


[Row(Ports_Array=['17128', '17136']),
 Row(Ports_Array=['17128', '17130', '17132', '17134', '17136', '17138', '17140']),
 Row(Ports_Array=['23', '80', '81', '1023', '2323', '5555', '7574', '8080', '8443', '37215', '49152', '52869'])]

In [37]:
Ports_list_RDD = Ports_Scanned_RDD.map(lambda row: row.Ports_Array )

In [38]:
Ports_list_RDD.take(3)

25/04/02 14:26:45 WARN BlockManager: Task 26 already completed, not releasing lock for rdd_99_0


[['17128', '17136'],
 ['17128', '17130', '17132', '17134', '17136', '17138', '17140'],
 ['23',
  '80',
  '81',
  '1023',
  '2323',
  '5555',
  '7574',
  '8080',
  '8443',
  '37215',
  '49152',
  '52869']]

In [39]:
flattened_Ports_list_RDD = Ports_list_RDD.flatMap(lambda x: x )

In [40]:
flattened_Ports_list_RDD.take(7)

25/04/02 14:26:47 WARN BlockManager: Task 27 already completed, not releasing lock for rdd_99_0


['17128', '17136', '17128', '17130', '17132', '17134', '17136']

In [41]:
Port_1_RDD = flattened_Ports_list_RDD.map(lambda x: (x,1))
Port_1_RDD.take(7)

25/04/02 14:26:50 WARN BlockManager: Task 28 already completed, not releasing lock for rdd_99_0


[('17128', 1),
 ('17136', 1),
 ('17128', 1),
 ('17130', 1),
 ('17132', 1),
 ('17134', 1),
 ('17136', 1)]

In [42]:
Port_count_RDD = Port_1_RDD.reduceByKey(lambda y,z: y+z, 5)

# Problem 2 (10%) 
### Complete The code below to find top k ports scanned by non-extreme multi-port scanners using ``sortByKey``, like mini-project 1.
### We will set k to 120 for mini-project 2.

In [43]:
Sorted_Count_Port_RDD = Port_count_RDD.map(lambda x: (x[1], x[0])).sortByKey( ascending = False)

In [44]:
top_ports = 120
Sorted_Count_Port_RDD.take(top_ports)

[(25272, '17132'),
 (25134, '17130'),
 (25117, '17140'),
 (25074, '17128'),
 (25040, '17138'),
 (24928, '17136'),
 (18183, '17134'),
 (18166, '17142'),
 (13376, '80'),
 (13248, '8080'),
 (13126, '23'),
 (5808, '2323'),
 (4850, '81'),
 (4091, '1023'),
 (4063, '5555'),
 (4026, '52869'),
 (3989, '8443'),
 (3933, '49152'),
 (3866, '7574'),
 (3860, '37215'),
 (3483, '54594'),
 (3088, '34218'),
 (3049, '34220'),
 (3036, '33962'),
 (3034, '33968'),
 (3033, '34224'),
 (3024, '34228'),
 (3008, '33960'),
 (2977, '33964'),
 (2957, '34216'),
 (2946, '33970'),
 (2942, '34226'),
 (2932, '33972'),
 (2347, '50401'),
 (1835, '34222'),
 (1792, '34230'),
 (1790, '33966'),
 (1692, '33974'),
 (1092, '445'),
 (1078, '0'),
 (647, '22'),
 (571, '8291'),
 (531, '8728'),
 (421, '1433'),
 (302, '8000'),
 (293, '8081'),
 (282, '5353'),
 (275, '2004'),
 (256, '11211'),
 (245, '6881'),
 (245, '443'),
 (241, '8082'),
 (240, '4000'),
 (238, '5060'),
 (236, '8083'),
 (224, '8088'),
 (213, '6379'),
 (191, '9527'),
 (18

In [47]:
path3="/storage/home/ajv5723/work/MiniProj2/local/sorted_top_ports_counts"
Sorted_Count_Port_RDD.saveAsTextFile(path3)

# Because we have applied ``persist()`` on ``NEMP_scanners_DF``, and the above action has generated the NEMP_scanners_DF, we can release the resource of ``non_extreme_multi_port_scanners`` because we don't need it in the rest of the code.

In [45]:
non_extreme_multi_port_scanners.unpersist()

DataFrame[_c0: int, id: int, numports: int, lifetime: decimal(10,0), Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: decimal(10,0), mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string]

# Like miniproject 1, we want to get a list of k top ports.  However, unlike miniproject 1, we use the list of k top ports to create One Hot Encoding for each top port in the list.
# We use the top k ports for One Hot Encoding
# The value of top_ports (120) is assigned in Problem 2

In [46]:
Sorted_Ports_RDD= Sorted_Count_Port_RDD.map(lambda x: x[1] )
Top_Ports_list = Sorted_Ports_RDD.take(top_ports)

In [47]:
Top_Ports_list

['17132',
 '17130',
 '17140',
 '17128',
 '17138',
 '17136',
 '17134',
 '17142',
 '80',
 '8080',
 '23',
 '2323',
 '81',
 '1023',
 '5555',
 '52869',
 '8443',
 '49152',
 '7574',
 '37215',
 '54594',
 '34218',
 '34220',
 '33962',
 '33968',
 '34224',
 '34228',
 '33960',
 '33964',
 '34216',
 '33970',
 '34226',
 '33972',
 '50401',
 '34222',
 '34230',
 '33966',
 '33974',
 '445',
 '0',
 '22',
 '8291',
 '8728',
 '1433',
 '8000',
 '8081',
 '5353',
 '2004',
 '11211',
 '6881',
 '443',
 '8082',
 '4000',
 '5060',
 '8083',
 '8088',
 '6379',
 '9527',
 '30301',
 '7001',
 '9200',
 '7002',
 '1027',
 '1900',
 '3389',
 '5900',
 '21',
 '6380',
 '88',
 '35',
 '8181',
 '5000',
 '389',
 '56880',
 '5001',
 '137',
 '8008',
 '7547',
 '49153',
 '4444',
 '139',
 '2222',
 '8001',
 '3544',
 '8888',
 '5984',
 '2480',
 '53',
 '1883',
 '873',
 '631',
 '9000',
 '50070',
 '161',
 '4786',
 '60001',
 '8090',
 '27017',
 '85',
 '12866',
 '3443',
 '111',
 '83',
 '82',
 '548',
 '5061',
 '995',
 '5901',
 '554',
 '10001',
 '9943',


In [48]:
len(Top_Ports_list)

120

#  A.2 One Hot Encoding of Top K Ports
## One-Hot-Encoded Feature/Column Name
Because we need to create a name for each one-hot-encoded feature, which is one of the top k ports, we can adopt the convention that the column name is "PortXXXX", where "XXXX" is a port number. This can be done by concatenating two strings using ``+``.

In [49]:
Top_Ports_list[1]

'17130'

In [50]:
FeatureName = "Port"+Top_Ports_list[1]

In [51]:
FeatureName

'Port17130'

## One-Hot-Encoding using withColumn and array_contains

# Problem 3 (10 points) Complete the code below for One-Hot-Encoding of the SECOND top port.

In [52]:
from pyspark.sql.functions import array_contains

In [53]:
NEMP_Scanners2_df= NEMP_Scanners_df.withColumn("Port"+Top_Ports_list[1], array_contains("Ports_Array",Top_Ports_list[1]))

In [54]:
NEMP_Scanners2_df.show(10)

+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+--------------------+--------------------+------------------------+--------------------+---------+
|    _c0|     id|numports|lifetime|Bytes|Packets|average_packetsize|MinUniqueDests|MaxUniqueDests|MinUniqueDest24s|MaxUniqueDest24s|average_lifetime|mirai| zmap|masscan|country|traffic_types_scanned_str|   ports_scanned_str|host_tags_per_censys|host_services_per_censys|         Ports_Array|Port17130|
+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+--------------------+--------------------+------------------------+--------------------+---------+
|2091467|2091467|       2|     200|  752|     12|                62|             1|           

## Verify the Correctness of One-Hot-Encoded Feature
## Problem 4 (5 points)
### Check whether one-hot encoding of the second top port is encoded correctly by completing the code below and enter your answer the in the next Markdown cell.

In [55]:
Second_top_port_scanners_count = NEMP_Scanners2_df.where(col("Port"+Top_Ports_list[1])== True).count()

In [56]:
print(Second_top_port_scanners_count)

25134


In [57]:
Sorted_Count_Port_RDD.take(3)

[(25272, '17132'), (25134, '17130'), (25117, '17140')]

## Answer for Problem 4:
- The second top port is : '17130'
- The total number of scanners that scan the secont top port, based on ``Sorted_Count_Port_RDD`` is: 25134
- Is this number the same as the number of scanners whose One-Hot-Encoded feature of the second top port is True? Yes it is. 25134.

## Generate Hot-One Encoded Feature for each of the top k ports in the Top_Ports_list

- Iterate through the Top_Ports_list so that each top port is one-hot encoded into the DataFrame for non-extreme multi-port scanners (i.e., `NEMP_Scanners2.df`).

## Problem 5 (10 points)
Complete the following PySpark code for encoding the top n ports using One Hot Encoding, where n is specified by the variable ```top_ports```

In [58]:
top_ports

120

In [59]:
Top_Ports_list[top_ports - 1]

'9999'

In [60]:
for i in range(0, top_ports):
    # "Port" + Top_Ports_list[i]  is the name of each new feature created through One Hot Encoding Top_Ports_list
    NEMP_Scanners3_df = NEMP_Scanners2_df.withColumn("Port" + Top_Ports_list[i], array_contains("Ports_Array",Top_Ports_list[i]))
    NEMP_Scanners2_df = NEMP_Scanners3_df

In [61]:
NEMP_Scanners2_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- id: integer (nullable = true)
 |-- numports: integer (nullable = true)
 |-- lifetime: decimal(10,0) (nullable = true)
 |-- Bytes: integer (nullable = true)
 |-- Packets: integer (nullable = true)
 |-- average_packetsize: integer (nullable = true)
 |-- MinUniqueDests: integer (nullable = true)
 |-- MaxUniqueDests: integer (nullable = true)
 |-- MinUniqueDest24s: integer (nullable = true)
 |-- MaxUniqueDest24s: integer (nullable = true)
 |-- average_lifetime: decimal(10,0) (nullable = true)
 |-- mirai: boolean (nullable = true)
 |-- zmap: boolean (nullable = true)
 |-- masscan: boolean (nullable = true)
 |-- country: string (nullable = true)
 |-- traffic_types_scanned_str: string (nullable = true)
 |-- ports_scanned_str: string (nullable = true)
 |-- host_tags_per_censys: string (nullable = true)
 |-- host_services_per_censys: string (nullable = true)
 |-- Ports_Array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |

# Problem 6 (10 points)
## Complete the code below to use k-means (number of clusters = 200) to cluster non-extreme multi-port scanners using one-hot-encoded top 120 ports.

## Specify Parameters for k Means Clustering

In [62]:
input_features = [ ]
for i in range(0, top_ports):
    input_features.append( "Port"+ Top_Ports_list[i] )

In [63]:
print(input_features)

['Port17132', 'Port17130', 'Port17140', 'Port17128', 'Port17138', 'Port17136', 'Port17134', 'Port17142', 'Port80', 'Port8080', 'Port23', 'Port2323', 'Port81', 'Port1023', 'Port5555', 'Port52869', 'Port8443', 'Port49152', 'Port7574', 'Port37215', 'Port54594', 'Port34218', 'Port34220', 'Port33962', 'Port33968', 'Port34224', 'Port34228', 'Port33960', 'Port33964', 'Port34216', 'Port33970', 'Port34226', 'Port33972', 'Port50401', 'Port34222', 'Port34230', 'Port33966', 'Port33974', 'Port445', 'Port0', 'Port22', 'Port8291', 'Port8728', 'Port1433', 'Port8000', 'Port8081', 'Port5353', 'Port2004', 'Port11211', 'Port6881', 'Port443', 'Port8082', 'Port4000', 'Port5060', 'Port8083', 'Port8088', 'Port6379', 'Port9527', 'Port30301', 'Port7001', 'Port9200', 'Port7002', 'Port1027', 'Port1900', 'Port3389', 'Port5900', 'Port21', 'Port6380', 'Port88', 'Port35', 'Port8181', 'Port5000', 'Port389', 'Port56880', 'Port5001', 'Port137', 'Port8008', 'Port7547', 'Port49153', 'Port4444', 'Port139', 'Port2222', 'Por

In [64]:
va = VectorAssembler().setInputCols(input_features).setOutputCol("features")

In [65]:
data= va.transform(NEMP_Scanners2_df)

In [66]:
data.show(1)

25/04/02 14:52:34 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+-----------------+--------------------+------------------------+--------------+---------+---------+---------+---------+---------+---------+---------+---------+------+--------+------+--------+------+--------+--------+---------+--------+---------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-------+-----+------+--------+--------+--------+--------+--------+--------+--------+---------+--------+-------+--------+--------+--------+--------+--------+--------+--------+---------+--------+--------+--------+--------+--------+--------+--------+------+--------+------+------+--------+--------+-------+---------+--------+-------+--------+--------+-----

In [67]:
data.persist()

DataFrame[_c0: int, id: int, numports: int, lifetime: decimal(10,0), Bytes: int, Packets: int, average_packetsize: int, MinUniqueDests: int, MaxUniqueDests: int, MinUniqueDest24s: int, MaxUniqueDest24s: int, average_lifetime: decimal(10,0), mirai: boolean, zmap: boolean, masscan: boolean, country: string, traffic_types_scanned_str: string, ports_scanned_str: string, host_tags_per_censys: string, host_services_per_censys: string, Ports_Array: array<string>, Port17130: boolean, Port17132: boolean, Port17140: boolean, Port17128: boolean, Port17138: boolean, Port17136: boolean, Port17134: boolean, Port17142: boolean, Port80: boolean, Port8080: boolean, Port23: boolean, Port2323: boolean, Port81: boolean, Port1023: boolean, Port5555: boolean, Port52869: boolean, Port8443: boolean, Port49152: boolean, Port7574: boolean, Port37215: boolean, Port54594: boolean, Port34218: boolean, Port34220: boolean, Port33962: boolean, Port33968: boolean, Port34224: boolean, Port34228: boolean, Port33960: boo

In [68]:
km = KMeans(featuresCol= "features", predictionCol="prediction").setK(200).setSeed(127)
km.explainParams()

'distanceMeasure: the distance measure. Supported options: \'euclidean\' and \'cosine\'. (default: euclidean)\nfeaturesCol: features column name. (default: features, current: features)\ninitMode: The initialization algorithm. This can be either "random" to choose random points as initial cluster centers, or "k-means||" to use a parallel variant of k-means++ (default: k-means||)\ninitSteps: The number of steps for k-means|| initialization mode. Must be > 0. (default: 2)\nk: The number of clusters to create. Must be > 1. (default: 2, current: 200)\nmaxIter: max number of iterations (>= 0). (default: 20)\npredictionCol: prediction column name. (default: prediction, current: prediction)\nseed: random seed. (default: 2106198337667939882, current: 127)\ntol: the convergence tolerance for iterative algorithms (>= 0). (default: 0.0001)\nweightCol: weight column name. If this is not set or empty, we treat all instance weights as 1.0. (undefined)'

In [69]:
kmModel=km.fit(data)

In [70]:
kmModel

KMeansModel: uid=KMeans_119661849517, k=200, distanceMeasure=euclidean, numFeatures=120

In [71]:
predictions = kmModel.transform(data)

In [72]:
predictions.show(3)

+-------+-------+--------+--------+-----+-------+------------------+--------------+--------------+----------------+----------------+----------------+-----+-----+-------+-------+-------------------------+--------------------+--------------------+------------------------+--------------------+---------+---------+---------+---------+---------+---------+---------+---------+------+--------+------+--------+------+--------+--------+---------+--------+---------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-------+-----+------+--------+--------+--------+--------+--------+--------+--------+---------+--------+-------+--------+--------+--------+--------+--------+--------+--------+---------+--------+--------+--------+--------+--------+--------+--------+------+--------+------+------+--------+--------+-------+---------+--------+-------+--------+-----

# Find The Size of The First Cluster

In [73]:
Cluster1_df=predictions.where(col("prediction")==0)

In [74]:
Cluster1_df.count()

540

In [75]:
summary = kmModel.summary

In [76]:
summary.clusterSizes

[540,
 1527,
 581,
 308,
 374,
 7189,
 1661,
 730,
 81,
 115,
 112,
 428,
 851,
 7204,
 66,
 369,
 77,
 285,
 446,
 54,
 24,
 100,
 344,
 826,
 143,
 127,
 271,
 46,
 371,
 252,
 226,
 78,
 592,
 742,
 105,
 519,
 2323,
 1139,
 869,
 251,
 256,
 886,
 528,
 741,
 159,
 188,
 387,
 208,
 271,
 181,
 394,
 485,
 90,
 889,
 375,
 234,
 78,
 422,
 109,
 146,
 298,
 832,
 366,
 117,
 369,
 106,
 142,
 92,
 140,
 175,
 104,
 1198,
 652,
 362,
 339,
 144,
 623,
 370,
 4,
 779,
 121,
 153,
 181,
 358,
 913,
 47,
 3,
 121,
 122,
 253,
 102,
 314,
 142,
 466,
 75,
 198,
 770,
 805,
 98,
 231,
 664,
 367,
 133,
 142,
 156,
 752,
 353,
 390,
 60,
 255,
 352,
 202,
 192,
 67,
 74,
 242,
 83,
 382,
 542,
 97,
 478,
 43,
 111,
 289,
 87,
 37,
 646,
 9,
 269,
 913,
 33,
 89,
 185,
 66,
 65,
 82,
 163,
 311,
 136,
 53,
 88,
 259,
 182,
 94,
 874,
 157,
 277,
 467,
 122,
 315,
 126,
 109,
 73,
 274,
 266,
 665,
 300,
 113,
 369,
 130,
 137,
 45,
 96,
 97,
 101,
 493,
 496,
 197,
 37,
 494,
 68,
 473,
 1

In [77]:
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)

In [78]:
print('Silhouette Score of the Clustering Result is ', silhouette)

Silhouette Score of the Clustering Result is  0.4994603446954778


In [79]:
centers = kmModel.clusterCenters()

In [80]:
print(centers)

[array([1.        , 1.        , 0.        , 1.        , 1.        ,
       1.        , 0.47037037, 0.40740741, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.00185185, 0.06111111, 0.03333333, 0.03333333, 0.01481481,
       0.03333333, 0.03148148, 0.02962963, 0.02407407, 0.03703704,
       0.        , 0.03888889, 0.01296296, 0.        , 0.01111111,
       0.02777778, 0.02222222, 0.01481481, 0.        , 0.00185185,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.    

# Record cluster index, cluster size, percentage of Mirai scanners, and cluster centers for each clusters formed.
## The value of cluster center for a OHE top port is the percentage of data/clusters in the cluster that scans the top port. For example, a cluster center `[0.094, 0.8, 0, ...]` indicates the following
- 9.4% of the scanners in the cluster scan Top_Ports_list[0]: port 17132
- 80% of the scanners in the cluster scan Top_Ports_list[1]: port 17130
- No scanners in the cluster scan Top_Ports_list[2]: port 17140

# Problem 7 (15 points) Complete the code below for computing the percentage of Mirai scanners for each scanner, and record it together with cluster centers for each cluster. Add persist to PySpark DataFrames that are used multiple times.  Add unpersist whenever the resource for a PySpark DataFrame is no longer needed.

In [81]:
import pandas as pd
import numpy as np
import math

In [84]:
# Define columns of the Pandas dataframe
column_list = ['cluster ID', 'size', 'mirai_ratio' ]
cluster_num =200
for feature in input_features:
    column_list.append(feature)
clusters_summary_df = pd.DataFrame( columns = column_list )
for i in range(0, cluster_num):
    cluster_i = predictions.where(col('prediction')==i)
    cluster_i_size = cluster_i.count()
    cluster_i_mirai_count = cluster_i.where(col("mirai")).count()
    cluster_i_mirai_ratio = cluster_i_mirai_count/cluster_i_size
    if cluster_i_mirai_count > 0:
        print("Cluster ", i, "; Mirai Ratio:", cluster_i_mirai_ratio, "; Cluster Size: ", cluster_i_size)
    cluster_row = [i, cluster_i_size, cluster_i_mirai_ratio]
    for j in range(0, len(input_features)):
        cluster_row.append(centers[i][j])
    clusters_summary_df.loc[i]= cluster_row

Cluster  5 ; Mirai Ratio: 0.0005564056196967589 ; Cluster Size:  7189
Cluster  9 ; Mirai Ratio: 0.09565217391304348 ; Cluster Size:  115
Cluster  12 ; Mirai Ratio: 0.0011750881316098707 ; Cluster Size:  851
Cluster  13 ; Mirai Ratio: 0.000694058856191005 ; Cluster Size:  7204
Cluster  14 ; Mirai Ratio: 0.24242424242424243 ; Cluster Size:  66


Cluster  27 ; Mirai Ratio: 0.7608695652173914 ; Cluster Size:  46
Cluster  37 ; Mirai Ratio: 0.9280070237050044 ; Cluster Size:  1139
Cluster  61 ; Mirai Ratio: 0.006009615384615385 ; Cluster Size:  832
Cluster  76 ; Mirai Ratio: 0.20224719101123595 ; Cluster Size:  623
Cluster  90 ; Mirai Ratio: 0.00980392156862745 ; Cluster Size:  102
Cluster  103 ; Mirai Ratio: 0.007042253521126761 ; Cluster Size:  142
Cluster  148 ; Mirai Ratio: 0.4426229508196721 ; Cluster Size:  122
Cluster  179 ; Mirai Ratio: 0.95 ; Cluster Size:  20


In [85]:
# Create a file name based on the number of top_ports
path4= "/storage/home/ajv5723/work/MiniProj2/local/MiraiRatio_Cluster_centers_"+"OHE"+ str(top_ports)+"top_ports"+"_k200.csv"
clusters_summary_df.to_csv(path4, header=True)

# Problem 8 (40 points)
- Modify the Jupyter Notebook for running in cluster mode using the big dataset (Day_2020_profile.csv). 
- Make sure you change the output directory from `../local/..` to `../cluster/..` so that it does not destroy the result you obtained in local mode.
- Add suitable persist and unpersist for Problem 7.
- If you want to compare performance of different persist options, make sure you change the output directory (e.g., ``../cluster_np/`` for without persist).
- Run the .py file the cluster mode. The following submission items (in addition to the completed Jupyter Notebook for local mode) are generated from the cluster mode.
- Submit the .py file 
- Submit the the log file that contains the run time information for a successful execution in the cluster mode.
- Submit the csv file that records the mirai percentage and cluster centers in the cluster mode.
- Submit the csv file that contains count of scanners that scan the same number of ports.
- Submit the first data file (part-00000) in ``sorted_top_ports_counts.txt``

In [86]:
ss.stop()